# Directed hyperedges & stoichiometry

A deeper dive than the tutorial. We build a small metabolic-style
network where each reaction is a *directed hyperedge* with explicit
substrates (head) and products (tail), then we set stoichiometric
coefficients on the incidence matrix and verify the column conserves
mass.

You'll see:

1. Building directed hyperedges
2. Reading them back via views
3. Reading and writing per-member coefficients on the incidence matrix
4. A small worked example (two coupled reactions sharing a metabolite)
5. Reversing the orientation of a reaction


In [1]:
from annnet import AnnNet

H = AnnNet(directed=False)  # the graph default doesn't matter; each edge sets its own
H.add_vertices(['Glc', 'ATP', 'G6P', 'ADP', 'F6P'])
H

AnnNet object with n_vertices × n_edges = 5 × 0
    directed: False
    slices: ['default']

## 1. Building directed hyperedges

`add_edges(src=[heads], tgt=[tails], edge_id=..., directed=True, weight=...)`.

We model two reactions:

- **hexokinase**: Glc + ATP → G6P + ADP
- **PGI**:        G6P → F6P


In [2]:
H.add_edges(src=['Glc', 'ATP'], tgt=['G6P', 'ADP'], edge_id='hexokinase', directed=True)
H.add_edges(src=['G6P'], tgt=['F6P'], edge_id='PGI', directed=True)

for eid in ('hexokinase', 'PGI'):
    rec = H._edges[eid]
    print(f'  {eid:>11}: head={set(rec.src)}, tail={set(rec.tgt)}, directed={rec.directed}')

   hexokinase: head={'ATP', 'Glc'}, tail={'ADP', 'G6P'}, directed=True
          PGI: head={'G6P'}, tail={'F6P'}, directed=True


## 2. Reading hyperedges via views

`G.views.edges()` exposes hyperedges through `head`, `tail`, and
`members` columns. `kind` is `'hyper'` for both directed and undirected.


In [3]:
H.views.edges().select(['edge_id', 'kind', 'head', 'tail', 'directed']).head()

edge_id,kind,head,tail,directed
str,str,list[str],list[str],bool
"""hexokinase""","""hyper""","[""ATP"", ""Glc""]","[""ADP"", ""G6P""]",true
"""PGI""","""hyper""","[""G6P""]","[""F6P""]",true


## 3. Stoichiometric coefficients

The incidence matrix is the source of truth. Each column is one edge;
each row is one entity. Default convention for directed hyperedges:

- `+w` at every **head** (substrate) row
- `-w` at every **tail** (product) row

You can override per-member by writing directly to the matrix.


In [4]:
# Read hexokinase's column off the incidence matrix
def print_column(graph, eid):
    col = graph._edges[eid].col_idx
    M = graph._matrix
    print(f'Column for {eid}:')
    for vid in graph.vertices():
        row = graph._entities[graph._resolve_entity_key(vid)].row_idx
        val = M[row, col]
        if val != 0:
            print(f'  {vid:>5}: {val:+.2f}')


print_column(H, 'hexokinase')

Column for hexokinase:
    Glc: +1.00
    ATP: +1.00
    G6P: -1.00
    ADP: -1.00


In [5]:
# Override the coefficient on F6P (toy example: assume a 2:1 reaction)
M = H._matrix
col = H._edges['PGI'].col_idx
row_F6P = H._entities[H._resolve_entity_key('F6P')].row_idx
M[row_F6P, col] = -2.0  # 2 molecules of F6P per reaction step (artificial)

print_column(H, 'PGI')

Column for PGI:
    G6P: +1.00
    F6P: -2.00


## 4. Mass balance for a metabolite shared between reactions

`G6P` is produced by hexokinase and consumed by PGI. If you assume both
reactions run at the same rate, the column-sum tells you the net flux
on each species.


In [6]:
import numpy as np

# Net flux at unit rate per reaction: B @ [1, 1] where B is the incidence
# matrix restricted to our two reactions.
cols = [H._edges[eid].col_idx for eid in ('hexokinase', 'PGI')]
B = np.asarray(H._matrix.tocsc()[:, cols].todense())

rates = np.array([1.0, 1.0])
flux = B @ rates  # net change per entity

for vid in H.vertices():
    row = H._entities[H._resolve_entity_key(vid)].row_idx
    print(f'  {vid:>5}: net flux = {flux[row]:+.2f}')

    Glc: net flux = +1.00
    ATP: net flux = +1.00
    G6P: net flux = +0.00
    ADP: net flux = -1.00
    F6P: net flux = -2.00


!!! note "Reading: convention is per-edge"
    Whether a reaction is **head→tail** or **tail→head** is a convention
    you set when calling `add_edges`. AnnNet stores both sides
    symmetrically in `rec.src` / `rec.tgt`. If you want the opposite
    convention, swap the lists.


## 5. Reading via `G.get_edge`

`get_edge(edge_id)` returns a typed `EdgeView` tuple. For directed
hyperedges, `view.source` is the head set, `view.target` is the tail
set, and `view.members` is the full incident set.


In [7]:
v = H.get_edge('hexokinase')
print('edge_id:', v.edge_id)
print('kind:   ', v.kind)
print('source: ', set(v.source))
print('target: ', set(v.target))
print('members:', set(v.members))
print('directed:', v.directed)

edge_id: hexokinase
kind:    hyper_directed
source:  {'ATP', 'Glc'}
target:  {'ADP', 'G6P'}
members: {'ATP', 'ADP', 'Glc', 'G6P'}
directed: True


## See also

- Tutorial `05_hyperedges.ipynb` for the introductory pass.
- The `notebooks/use_cases/uc2_multilayer_signaling_metabolism.ipynb`
  notebook for a real metabolic network built from a Human-GEM SBML file.
